In [2]:
# Cell 1 - Imports and Apple Silicon device detection
from pathlib import Path
from typing import List
import os
import numpy as np
from skimage import io
from tqdm import tqdm
import torch
from cellpose import models

def has_mps() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

device = torch.device("mps") if has_mps() else torch.device("cpu")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
print("torch:", torch.__version__)
print("device:", device)




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


torch: 2.7.1
device: mps


In [3]:
# Cell 2 — Project paths and parameters
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Inputs: expect data/red, data/yellow, data/blue, data/green
img_root = project_root / "data"
channels: List[str] = ["red", "yellow", "blue", "green"]

# Outputs: masks will go to analysis/cellpose_results2/<channel>/png/*_masks.png
masks_root = project_root / "analysis" / "cellpose_results2"

print("- project root is:" , project_root)
print("- image root is:" , img_root)
print("- channels are:", channels)
print("- masks root is:" , masks_root)

# Image extensions to search
image_exts = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]

# Segmentation parameters — IMPORTANT: diameter=None (not 0)
diameter = None
flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 64
batch_size = 4
debug_max_images = None  # set small int to smoke test



- project root is: /Users/ashi/github/cm4ai_codefest2025
- image root is: /Users/ashi/github/cm4ai_codefest2025/data
- channels are: ['red', 'yellow', 'blue', 'green']
- masks root is: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2


In [ ]:
# --- SMOKE TEST: find working Cellpose params on ONE yellow image (Cellpose 4.x) ---
from pathlib import Path
import os, sys, math
import numpy as np
from skimage import io
from skimage.measure import label
from cellpose import models

# 0) Inputs (assumes you already set img_root, device, and optionally pretrained_model)
ypath = img_root / "yellow"
if not ypath.exists():
    raise FileNotFoundError(f"Yellow dir not found: {ypath}")

# 1) pick first yellow image automatically
candidates = []
for ext in (".tif",".tiff",".png",".jpg",".jpeg"):
    candidates += sorted(ypath.glob(f"*{ext}"))
if not candidates:
    raise FileNotFoundError(f"No yellow images found under {ypath}")

test_path = candidates[0]
img = io.imread(test_path)

print(f"[SMOKE] Testing on: {test_path.name}")
print(" shape:", img.shape, "dtype:", img.dtype, "min..max:", float(img.min()), float(img.max()))
pcts = np.percentile(img, [0,1,5,50,95,99,100]).astype(float)
print(" percentiles [0,1,5,50,95,99,100]:", pcts.tolist())

# 2) determine channel_axis (RGB vs 2D)
is_rgb = (img.ndim == 3)
channel_axis_val = -1 if is_rgb else None
print(" channel_axis:", channel_axis_val, "(RGB)" if is_rgb else "(grayscale)")

def count_masks(arr: np.ndarray) -> int:
    if arr.size == 0:
        return 0
    if arr.dtype == bool:
        return int(np.max(label(arr)))
    # Cellpose labeled masks are int images where max label == #objects
    return int(np.max(arr))

# 3) define parameter grid
# respect your current choice if pretrained_model exists; otherwise start with cyto2
try:
    _default_model = pretrained_model
except NameError:
    _default_model = "cyto2"

models_to_try = [_default_model, "cyto3", "cpsam"]  # nuclei usually not for cytoplasm IF

diameters_to_try   = [None, 20, 30, 40, 60]
cellprobs_to_try   = [-2.0, 0.0, 0.2]     # permissive → default → stricter
flows_to_try       = [0.4, 0.9]
invert_to_try      = [False, True]        # sometimes “cells bright on dark” vs “dark on bright”
normalize_to_try   = [None, {"tile_norm_blocksize": 64}]

# 4) sweep
results = []  # (num_masks, model, diam, cellprob, flow, invert, norm_tag)
for mdl in models_to_try:
    tmp_model = models.CellposeModel(gpu=False, pretrained_model=mdl, device=device)
    for d in diameters_to_try:
        d_val = None if (d is None or d == 0) else float(d)
        for cp in cellprobs_to_try:
            for fl in flows_to_try:
                for inv in invert_to_try:
                    for norm in normalize_to_try:
                        try:
                            masks, flows, styles = tmp_model.eval(
                                [img],
                                diameter=d_val,                   # None or positive float
                                batch_size=1,
                                channel_axis=channel_axis_val,    # -1 for RGB, None for 2D
                                invert=inv,
                                flow_threshold=fl,
                                cellprob_threshold=cp,
                                normalize=norm,                   # None or tile norm
                            )
                            n = count_masks(masks[0])
                            tag = "none" if norm is None else "tile64"
                            results.append((n, mdl, d_val, cp, fl, inv, tag))
                        except Exception as e:
                            # mark as failure
                            results.append((-1, mdl, d_val, cp, fl, inv, "ERR"))

# 5) show top candidates
results_sorted = sorted(results, key=lambda x: x[0], reverse=True)
print("\nTop parameter combos (num_masks, model, diameter, cellprob, flow, invert, norm):")
for row in results_sorted[:10]:
    print(row)
    
# 6) pick winner (first with n > 0)
winner = next((r for r in results_sorted if r[0] > 0), None)
if winner:
    n, mdl, d_val, cp, fl, inv, tag = winner
    print("\nWINNER:")
    print(f" num_masks={n}, model='{mdl}', diameter={d_val}, "
          f"cellprob_threshold={cp}, flow_threshold={fl}, invert={inv}, normalize={tag}")

    # Build normalize argument string safely
    normalize_str = "None" if tag == "none" else "{'tile_norm_blocksize': 64}"

    # Print ready-to-paste production snippet
    print("\nPaste into production cell:")
    print(f"pretrained_model = '{mdl}'")
    print(f"diameter = {repr(d_val)}")
    print(f"cellprob_threshold = {cp}")
    print(f"flow_threshold = {fl}")
    print(f"invert = {inv}")
    print(f"normalize_arg = {normalize_str}")
else:
    print("\nNo masks found. Options:")
    print(" - Try larger diameters: 80, 100")
    print(" - Force invert the image (invert=True)")
    print(" - Ensure the channel has signal (check percentiles above)")
    print(" - Try a different model: 'cyto2'/'cyto3'/'cpsam'")

pretrained model /Users/ashi/.cellpose/models/cpsam not found, using default model


[SMOKE] Testing on: B2AI_1_Paclitaxel_A1_R2_z01_yellow.jpg
 shape: (2048, 2048, 3) dtype: uint8 min..max: 0.0 255.0
 percentiles [0,1,5,50,95,99,100]: [0.0, 0.0, 0.0, 5.0, 32.0, 48.0, 255.0]
 channel_axis: -1 (RGB)


Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
